# Multi-node PyTorch Lightning Fabric on Modal

**What Lightning Fabric is.**
[Fabric](https://lightning.ai/docs/fabric/stable/) is Lightning's
"lightweight" API. Unlike the full `LightningModule` / `Trainer`
stack (which manages the training loop for you), Fabric keeps
**your own loop** and just injects the distributed setup —
rank/world-size, device placement, gradient sync, precision
casting. You get DDP / FSDP / DeepSpeed strategies without
rewriting your loop; you give up the Trainer's callbacks, logging
hooks, and checkpointing helpers.

Pick Fabric when you want explicit control of the loop (research,
custom schedules); pick the Trainer when you want the full
framework.

**What this tutorial does.** Runs the tiny Transformer-on-WikiText2
demo from the Lightning docs on a 2-node × 8×H100 Modal cluster with
`strategy="ddp"` and `precision="bf16-mixed"`. No dataset / model /
W&B containers — the script downloads WikiText2 itself and logs to
stdout. This is the smallest Fabric run that still exercises the
clustered multi-node path.

For the shared primitives and the framework factory pattern see
[`quickstart`](../../intro/quickstart/quickstart.ipynb).

**What to watch.** stdout — `fabric.print` only prints on rank 0.
Loss should drop within the first iteration (the max_steps is set
to 1 for a smoke test; bump it for a real run).

In [ ]:
! pip install -q git+https://github.com/modal-projects/training-gym.git@joy/initial-setup

In [ ]:
import modal

from modal_training_gym.frameworks.lightning import (
    LightningConfig,
    LightningFrameworkConfig,
)

## Training script

Shipped as a source string (same pattern as
[`starcoder_llama2_7b`](../../sft/starcoder_llama2_7b/starcoder_llama2_7b.ipynb)).
The script itself has no multi-node specifics — Fabric handles
rank / world-size / device placement. `fabric.setup(model,
optimizer)` wraps them for distributed training; `fabric.backward()`
replaces the usual `loss.backward()` to make gradient sync
explicit.

In [ ]:
import textwrap

TRAIN_SCRIPT = textwrap.dedent(r'''
    import lightning as L
    import torch
    import torch.nn.functional as F
    from lightning.pytorch.demos import Transformer, WikiText2
    from torch.utils.data import DataLoader


    def main():
        L.seed_everything(42)

        fabric = L.Fabric()
        fabric.launch()

        with fabric.rank_zero_first():
            dataset = WikiText2(download=True)

        train_dataloader = DataLoader(dataset, batch_size=20, shuffle=True)
        model = Transformer(vocab_size=dataset.vocab_size)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

        model, optimizer = fabric.setup(model, optimizer)
        train_dataloader = fabric.setup_dataloaders(train_dataloader)

        max_steps = 1
        for batch_idx, batch in enumerate(train_dataloader):
            if batch_idx >= max_steps:
                break
            input, target = batch
            output = model(input, target)
            loss = F.nll_loss(output, target.view(-1))
            fabric.backward(loss)
            optimizer.step()
            optimizer.zero_grad()
            fabric.print(f"iter {batch_idx}/{max_steps} - loss {loss.item():.4f}")


    if __name__ == "__main__":
        main()
''').lstrip("\n")

## Experiment config

`LightningFrameworkConfig` carries the three Fabric knobs that
matter at launch time:

- `strategy="ddp"` — distributed data parallel. Swap to `"fsdp"`
  for FSDP sharding, or `"deepspeed"` for DeepSpeed.
- `precision="bf16-mixed"` — BF16 activations with FP32 master
  weights.
- `accelerator="gpu"` — trivially true here but required.

No `dataset` / `model` / `wandb` containers because the script
downloads WikiText2 itself and doesn't log externally.

In [ ]:
lightning_framework_config = LightningFrameworkConfig(
    gpu="H100",
    n_nodes=2,
    gpus_per_node=8,
    train_script_source=TRAIN_SCRIPT,
    accelerator="gpu",
    strategy="ddp",
    precision="bf16-mixed",
)

my_training_run = LightningConfig(
    framework_config=lightning_framework_config,
)

## Build and run

In [ ]:
app = my_training_run.build_app()

Interactive — one stage per cell:

In [ ]:
with app.run():
    app.download_dataset.remote()

In [ ]:
with app.run():
    app.upload_script.remote()

In [ ]:
with app.run():
    app.train.remote()